## Author: Kenny Loh Kit Yi
---

In [1]:
import os
import shutil
import subprocess
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from classes.data_cleaner import DataCleaner
from classes.dataframe_saver import DataFrameSaver
from classes.entity_processor import EntityProcessor
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, concat

In [2]:
# Delete old HDFS version
subprocess.run(['hdfs', 'dfs', '-rm', 'cleaned_articles1/*'])
subprocess.run(['hdfs', 'dfs', '-rm', 'entity/*'])
subprocess.run(['hdfs', 'dfs', '-rm', 'final_dictionary/*'])

rm: `cleaned_articles1/*': No such file or directory
rm: `entity/*': No such file or directory
rm: `final_dictionary/*': No such file or directory


CompletedProcess(args=['hdfs', 'dfs', '-rm', 'final_dictionary/*'], returncode=1)

In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("EntityAnalysis").master("local").getOrCreate()
input_path = "articles/articles.csv"
output_dir = "/home/student/de-assgt/content"
output_dir1 = "/home/student/de-assgt/content/final_dictionary"
file_name = "cleaned_articles1.csv"
file_name1 = 'entity.csv'
file_name2 = "final_dictionary.csv"
hdfs_path = "cleaned_articles1"
hdfs_path1 ='entity'
hdfs_path2 = 'final_dictionary'

24/12/22 20:08:28 WARN Utils: Your hostname, LAPTOP-PFPL3CLD. resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/12/22 20:08:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/22 20:08:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/12/22 20:08:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
# Read the CSV file from the local file system
df = DataCleaner.csvToDataFrame(input_path, spark)

df1 = DataCleaner.remove_numbers(df, column="article")
df1 = DataCleaner.remove_standalone_characters(df1, column="article")
df1 = DataCleaner.remove_words_in_parentheses(df1, column="article")

# Remove delimiters
delimiters_to_remove = r',.|;$&?!():@\'\"\[\]\<\>/{/}/‘/’/“/”'
df1 = DataCleaner.remove_delimiter(df1, column="article", delimiters=delimiters_to_remove)

# Conditional delimiter removal
delimiters_to_remove = "-'"
df1 = DataCleaner.smart_remove_delimiter(df1, column="article", delimiters=delimiters_to_remove)
df1 = DataCleaner.remove_hyphen_from_prefix(df1, column="article")

# Normalize spaces
df1 = DataCleaner.normalize_spaces(df1, column="article")

# Split 'article' column into words 
rdd1 = df1.rdd.flatMap(lambda line: line['article'].split(" ")) 
df_split1 = rdd1.map(lambda word: (word,)).toDF(["article"])  

DataFrameSaver.save_to_csv(df_split1, output_dir, file_name)

print(f"Cleaned article content saved as {file_name} in {output_dir}")
df_split1.show()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: hdfs://localhost:9000/user/student/articles/articles.csv.

In [ ]:
# Save DataFrame to HDFS
df_split1.coalesce(1).write.csv(hdfs_path, header=True, mode="overwrite")
print(f"Article content saved to HDFS at: {hdfs_path}")

In [ ]:
# Rename part-00000 file
subprocess.run(['hdfs', 'dfs', '-mv', 'cleaned_articles1/part-00000*', 'cleaned_articles1/cleaned_articles1.csv'])

In [ ]:
from pyspark.sql.functions import col
from classes.entity_processor import EntityProcessor

# Collect words that contain at least one uppercase letter
df_filtered = df_split1.filter(col("article").rlike(r'[A-Z]'))
words = [row["article"] for row in df_filtered.collect()]

# Create a set to store unique results
unique_results = set()
processed_phrases = set()
max_append_length = 5  
batch_size = 20  

# Process single words 
for i in range(0, len(words), batch_size):
    batch_words = words[i:i + batch_size]
    batch_titles = [EntityProcessor.normalize_word(word) for word in batch_words if word not in processed_phrases]
    processed_phrases.update(batch_titles)

    batch_results = EntityProcessor.get_entity_summaries(batch_titles)

    for title, (entity_type, summary, _) in batch_results.items():
        if summary:
            unique_results.add((entity_type, title, summary))

# Process multi-word 
phrases = EntityProcessor.generate_phrases(words, max_append_length)
for phrase in phrases:
    if phrase not in processed_phrases:
        batch_results = EntityProcessor.get_entity_summaries([phrase])
        for title, (entity_type, summary, _) in batch_results.items():
            if summary:
                unique_results.add((entity_type, title, summary))
                processed_phrases.add(phrase)

unique_results_list = list(unique_results)

df_results = spark.createDataFrame(unique_results_list, ["entity_type", "words", "definition"])

DataFrameSaver.save_to_csv_1(df_results, output_dir, file_name1)

In [ ]:
# Save DataFrame to HDFS
df_results.coalesce(1).write.csv(hdfs_path1, header=True, mode="overwrite")
print(f"Content saved to HDFS at: {hdfs_path1}")

In [ ]:
# Rename part-00000 file
subprocess.run(['hdfs', 'dfs', '-mv', 'entity/part-00000*', 'entity/entity.csv'])

In [ ]:
# Read the dictionary4 file
dictionary4_path = "dictionary4/dictionary4.csv"
df_articles = spark.read.option("quote", "\"").option("escape", "\"").csv(dictionary4_path, header=True, inferSchema=True)

# Read the entity.csv file
entity_file_path = "entity/entity.csv"
df_entity = spark.read.option("quote", "\"").option("escape", "\"").csv(entity_file_path, header=True, inferSchema=True)

# Add missing columns with null values from df_entity to df_articles
for col_name in set(df_entity.columns) - set(df_articles.columns):
    df_articles = df_articles.withColumn(col_name, F.lit("-"))

# Add missing columns with null values from df_articles to df_entity 
for col_name in set(df_articles.columns) - set(df_entity.columns):
    df_entity = df_entity.withColumn(col_name, F.lit("-"))

# Ensure that the columns in df_articles come first, followed by the columns in df_entity
df_articles_columns = df_articles.columns
df_entity_columns = df_entity.columns

# Rearrange columns so that df_articles columns come first followed by df_entity columns
df_entity = df_entity.select(df_entity_columns)

# Combine both DataFrames by union, keeping the order
df_merged = df_articles.unionByName(df_entity)

df_no_duplicates = df_merged.dropDuplicates()

DataFrameSaver.save_to_csv_1(df_no_duplicates, output_dir1, file_name2)

In [ ]:
df_merged.show()

In [ ]:
df_merged.coalesce(1).write.csv(hdfs_path2, header=True, mode="overwrite")
print(f"Article content saved to HDFS at: {hdfs_path2}")

In [ ]:
# Rename part-00000 file to something readable
subprocess.run(['hdfs', 'dfs', '-mv', 'final_dictionary/part-00000*', 'final_dictionary/final_dictionary.csv'])

In [ ]:
spark.stop()